# Nhóm 07 — Khai phá dữ liệu tuyển dụng khối nghề phần mềm

## Giai đoạn 1: Giữ nguyên bản cào mới nhất
Chọn bản cào mới nhất theo timestamp trong tên và sao chép thành `raw_jobs_data.csv`, giữ nguyên dữ liệu. Chạy lại sẽ ghi đè bản sao này, không sửa file cào nguồn.

In [157]:
from pathlib import Path
import shutil
import re
from bs4 import BeautifulSoup
from stopwordsiso import stopwords

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INPUT_FILE = max(RAW_DIR.glob("raw_data_????????_??????.csv"), key=lambda p: p.name)
RAW_PATH = PROJECT_ROOT / "notebooks" / "raw_jobs_data.csv"
shutil.copyfile(INPUT_FILE, RAW_PATH)
print(f"Đã sao chép {INPUT_FILE.name} --> {RAW_PATH.name}")

Đã sao chép raw_data_20261115_014436.csv --> raw_jobs_data.csv


## Giai đoạn 2: THỰC THI QUY TRÌNH KHAI PHÁ DỮ LIỆU TRÊN DỮ LIỆU THỰC TẾ
### Bước 2.1: Tiền xử lý và Làm sạch dữ liệu tuyển dụng

In [158]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [159]:
# Đọc file raw_jobs_data.csv vào DataFrame
df = pd.read_csv(RAW_PATH)
print(df.shape)
df.head(10)

(867, 11)


,job_title,company_name,location_raw,salary_raw,experience_raw,description_raw,requirements_raw,benefits_raw,level_raw,source_url,scraped_at
0,Chuyên Viên Kiểm Tra Chất Lượng Phần Mềm - Tester,CÔNG TY TNHH P.A VIỆT NAM,"Hồ Chí Minh: 344 Huỳnh Tấn Phát, Phường Tân Th...",10 - 20 triệu,1 năm,"<div class=""box-job-information-detail-item__t...","<h2 class=""box-job-information-detail-item__ti...","<h2 class=""box-job-information-detail-item__ti...",Nhân viên,https://www.topcv.vn/viec-lam/chuyen-vien-kiem...,2026-11-15T01:45:11+07:00
1,Quản Lý Dự Án Hạ Tầng Facilities - PM,SVTECH,"Hồ Chí Minh: 2A Phan Thúc Duyện, Phường Tân Sơ...",Thoả thuận,3 năm,"<div class=""box-job-information-detail-item__t...","<h2 class=""box-job-information-detail-item__ti...","<h2 class=""box-job-information-detail-item__ti...",Nhân viên,https://www.topcv.vn/viec-lam/quan-ly-du-an-ha...,2026-11-15T01:45:23+07:00
2,Chuyên Viên Kiểm Thử Phần Mềm,TTI JSC,"Hồ Chí Minh: 370 Trường Chinh, Phường Tân Bình...",10 - 15 triệu,2 năm,"<div class=""box-job-information-detail-item__t...","<h2 class=""box-job-information-detail-item__ti...","<h2 class=""box-job-information-detail-item__ti...",Nhân viên,https://www.topcv.vn/viec-lam/chuyen-vien-kiem...,2026-11-15T01:45:37+07:00
3,Fullstack Developer,Tập đoàn FPT,"Hà Nội: toà FPT, số 10 Phạm Văn Bạch, Cầu Giấy...",Thoả thuận,2 năm,"<div class=""box-job-information-detail-item__t...","<h2 class=""box-job-information-detail-item__ti...","<h2 class=""box-job-information-detail-item__ti...",Nhân viên,https://www.topcv.vn/viec-lam/fullstack-develo...,2026-11-15T01:45:49+07:00
4,QA Engineer (Pre-Deployment Codebase Testing),CÔNG TY CỔ PHẦN ĐẦU TƯ VÀ PHÁT TRIỂN NĂNG LƯỢN...,Hồ Chí Minh: Phường Bình Trưng (Phường Bình Tr...,Thoả thuận,3 năm,"<div class=""box-job-information-detail-item__t...","<h2 class=""box-job-information-detail-item__ti...","<h2 class=""box-job-information-detail-item__ti...",Nhân viên,https://www.topcv.vn/viec-lam/qa-engineer-pre-...,2026-11-15T01:46:01+07:00
5,Fullstack Developer ( NodeJS / PHP /Go) – E-Co...,XIPAT FLEXIBLE SOLUTIONS COMPANY LIMITED,"Hà Nội: Số 18 ngõ 11 Thái Hà, Phường Đống Đa (...",Thoả thuận,5 năm,"<div class=""box-job-information-detail-item__t...","<h2 class=""box-job-information-detail-item__ti...","<h2 class=""box-job-information-detail-item__ti...",Nhân viên,https://www.topcv.vn/viec-lam/fullstack-develo...,2026-11-15T01:46:13+07:00
6,Nhân Viên Tư Vấn Triển Khai Phần Mềm Website (...,CÔNG TY TNHH CÔNG NGHỆ PHẦN MỀM NASANI,Hồ Chí Minh: (Tất cả các phường) - Phạm vi làm...,7 - 20 triệu,Không yêu cầu,"<div class=""box-job-information-detail-item__t...","<h2 class=""box-job-information-detail-item__ti...","<h2 class=""box-job-information-detail-item__ti...",Nhân viên,https://www.topcv.vn/viec-lam/nhan-vien-tu-van...,2026-11-15T01:46:25+07:00
7,Java Developer,"Cathay Insurance Co., Ltd","Hồ Chí Minh: Lim Tower 3, 29A Nguyễn Đình Chiể...",Thoả thuận,1 năm,"<div class=""box-job-information-detail-item__t...","<h2 class=""box-job-information-detail-item__ti...","<h2 class=""box-job-information-detail-item__ti...",Nhân viên,https://www.topcv.vn/viec-lam/java-developer/8...,2026-11-15T01:46:37+07:00
8,Tester (Manual/Automation) – Thu Nhập 18-24 Tr...,Công ty TNHH Thiết kế T&T,"Hà Nội: Tòa nhà VPI 167 Trung Kính, Phường Yên...",18 - 24 triệu,2 năm,"<div class=""box-job-information-detail-item__t...","<h2 class=""box-job-information-detail-item__ti...","<h2 class=""box-job-information-detail-item__ti...",Nhân viên,https://www.topcv.vn/viec-lam/tester-manual-au...,2026-11-15T01:46:48+07:00
9,Nhân Viên Tư Vấn Triển Khai Phần Mềm Website (...,CÔNG TY TNHH CÔNG NGHỆ PHẦN MỀM NASANI,Hồ Chí Minh: (Tất cả các phường) - Phạm vi làm...,7 - 20 triệu,Không yêu cầu,"<div class=""box-job-information-detail-item__t...","<h2 class=""box-job-information-detail-item__ti...","<h2 class=""box-job-information-detail-item__ti...",Nhân viên,https://www.topcv.vn/viec-lam/nhan-vien-tu-van...,2026-11-15T01:47:01+07:00


===> Dữ liệu thô có 867 dòng và 11 cột

In [160]:
df.describe()

,job_title,company_name,location_raw,salary_raw,experience_raw,description_raw,requirements_raw,benefits_raw,level_raw,source_url,scraped_at
count,867,867,867,867,867,867,867,867,867,867,867
unique,770,467,629,174,8,851,863,820,6,867,867
top,Product Owner,Công ty Cổ phần Công nghệ Phần mềm Tinh Vân,Hà Nội: Phường Yên Hòa (Phường Yên Hòa thuộc q...,Thoả thuận,3 năm,"<div class=""box-job-information-detail-item__t...","<h2 class=""box-job-information-detail-item__ti...","<h2 class=""box-job-information-detail-item__ti...",Nhân viên,https://www.topcv.vn/viec-lam/chuyen-vien-kiem...,2026-11-15T01:45:11+07:00
freq,13,18,9,430,217,2,2,4,550,1,1


In [161]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 867 entries, 0 to 866
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   job_title         867 non-null    str  
 1   company_name      867 non-null    str  
 2   location_raw      867 non-null    str  
 3   salary_raw        867 non-null    str  
 4   experience_raw    867 non-null    str  
 5   description_raw   867 non-null    str  
 6   requirements_raw  867 non-null    str  
 7   benefits_raw      867 non-null    str  
 8   level_raw         867 non-null    str  
 9   source_url        867 non-null    str  
 10  scraped_at        867 non-null    str  
dtypes: str(11)
memory usage: 74.6 KB


==> Bộ dữ liệu không có ô nào null

In [162]:
salary_col = df['salary_raw'].copy()
salary = salary_col.astype('string').str.strip().str.lower()

number_pattern = r'\d+(?:[.,]\d+)*'
amount_pattern = rf'(?:(?:usd|\$)\s*)?{number_pattern}\s*(?:triệu|tr|m|usd|\$)?'

# Phân loại dạng lương
salary_patterns = {
    'Thỏa thuận': r'th(?:ỏa|oả) thuận',
    'Khoảng lương': rf'{amount_pattern}\s*[-–—]\s*{amount_pattern}',
    'Cận trên': rf'(?:tới|up\s*to)\s*{amount_pattern}',
    'Cận dưới': rf'(?:từ|from)\s*{amount_pattern}',
    'Mức cụ thể': rf'{amount_pattern}',
}

salary_type = pd.Series('Khác', index=salary.index)
for label, pattern in salary_patterns.items():
    salary_type.loc[salary.str.fullmatch(pattern, na=False)] = label

# Phân loại đơn vị
salary_unit = pd.Series('Không ghi đơn vị', index=salary.index)
salary_unit.loc[salary.str.contains(r'(?:triệu|tr|m)$', na=False)] = 'Triệu'
salary_unit.loc[salary.str.contains(r'usd|\$', na=False)] = 'USD / $'
salary_unit.loc[salary_type.eq('Thỏa thuận')] = 'Không áp dụng'

thieu = salary.isna() | salary.eq('').fillna(False)
salary_type.loc[thieu] = 'Thiếu'
salary_unit.loc[thieu] = 'Không áp dụng'

# Bảng thống kê: Loại | Triệu | USD / $ | ... | Tổng
df_stats = pd.crosstab(salary_type, salary_unit).reindex(
    index=[*salary_patterns.keys(), 'Thiếu', 'Khác'],
    columns=['Triệu', 'USD / $', 'Không ghi đơn vị', 'Không áp dụng'],
    fill_value=0
)
df_stats['Tổng'] = df_stats.sum(axis=1)
df_thong_ke = df_stats.rename_axis('Loại').reset_index()

display(df_thong_ke)

col_0,Loại,Triệu,USD / $,Không ghi đơn vị,Không áp dụng,Tổng
0,Thỏa thuận,0,0,0,430,430
1,Khoảng lương,334,6,0,0,340
2,Cận trên,86,5,0,0,91
3,Cận dưới,6,0,0,0,6
4,Mức cụ thể,0,0,0,0,0
5,Thiếu,0,0,0,0,0
6,Khác,0,0,0,0,0


===> Sau khảo sát cột `Muc_Luong_Goc` cho thấy cột dữ liệu này có `4` loại lương gồm `thỏa thuận`, `khoảng lương`, `cận trên`, `cận dưới`, trong đó:
- `Khoảng lương` có `334` loại `tiền Việt` và `6` loại `tiền Đô`
- `Cận trên` có `86` loại `tiền Việt` và `5` loại `tiền Đô`
- `Cận dưới` có `6` loại `tiền Việt` và `không có` loại `tiền Đô`

In [163]:
# Chọn tỉ giá USD sang VND để quy đổi các mức lương USD sang VND
usd_exchange_rate = 26000 

df_clean = df.copy()
# Kiểm tra các dòng trùng lặp dựa trên tất cả các cột ngoại trừ source_url và scraped_at
content_columns = df_clean.columns.difference(
    ["source_url", "scraped_at"]
)

duplicates = df_clean.duplicated(subset=content_columns)
display(df_clean.loc[duplicates, ["job_title", "source_url"]])

df_clean = df_clean.loc[~duplicates].copy()

# Format lại cột salary_raw
def parse_salary(text):
    text = text.strip().lower()

    if "thỏa thuận" in text or "thoả thuận" in text:
        return None, None

    is_usd = "usd" in text or "$" in text
    multiplier = usd_exchange_rate if is_usd else 1_000_000

    numbers = re.findall(r'\d+(?:[.,]\d+)*', text)
    numbers = [
        float(so.replace(",", "") if is_usd else so.replace(",", "."))
        * multiplier
        for so in numbers
    ]

    if text.startswith(("tới", "up to", "upto")):
        return None, numbers[0]

    if text.startswith(("từ", "from")):
        return numbers[0], None

    return numbers[0], numbers[-1]

df_clean['salary_type'] = salary_type

df_clean[['salary_min', 'salary_max']] = pd.DataFrame(
    df_clean['salary_raw'].apply(parse_salary).tolist(),
    index=df_clean.index,
    dtype='float64'
)

display(df_clean[['salary_raw', 'salary_min', 'salary_max','salary_type']].head(10))

,job_title,source_url
427,Trưởng Phòng IT ( Công Nghệ Thông Tin ),https://www.topcv.vn/viec-lam/truong-phong-it-...


,salary_raw,salary_min,salary_max,salary_type
0,10 - 20 triệu,10000000.0,20000000.0,Khoảng lương
1,Thoả thuận,NaN,NaN,Thỏa thuận
2,10 - 15 triệu,10000000.0,15000000.0,Khoảng lương
3,Thoả thuận,NaN,NaN,Thỏa thuận
4,Thoả thuận,NaN,NaN,Thỏa thuận
5,Thoả thuận,NaN,NaN,Thỏa thuận
6,7 - 20 triệu,7000000.0,20000000.0,Khoảng lương
7,Thoả thuận,NaN,NaN,Thỏa thuận
8,18 - 24 triệu,18000000.0,24000000.0,Khoảng lương
9,7 - 20 triệu,7000000.0,20000000.0,Khoảng lương


===> Thực hiện chuẩn hóa cột `salary_raw` thành hai cột `salary_min`, `salary_max`, quy đổi về VNĐ theo tỷ giá đã chọn. Gán nhãn `salary_type` gồm 4 loại xuất hiện trong dữ liệu: `Thỏa thuận`, `Khoảng lương`, `Cận trên`, `Cận dưới`. Những cận không xác định được giữ là giá trị thiếu (NaN).

In [164]:
df['location_raw'].head(5)

0    Hồ Chí Minh: 344 Huỳnh Tấn Phát, Phường Tân Th...
1    Hồ Chí Minh: 2A Phan Thúc Duyện, Phường Tân Sơ...
2    Hồ Chí Minh: 370 Trường Chinh, Phường Tân Bình...
3    Hà Nội: toà FPT, số 10 Phạm Văn Bạch, Cầu Giấy...
4    Hồ Chí Minh: Phường Bình Trưng (Phường Bình Tr...
Name: location_raw, dtype: str

===> Khảo sát sơ qua địa chỉ của cột `location_raw` hầu hết đều có dạng `Tỉnh thành: Địa chỉ`. Bước tiếp theo sẽ chuẩn hóa cột `location_raw` về tỉnh thành

In [165]:
df_clean['province'] = df_clean['location_raw'].str.split(':').str[0].str.strip()
print(df_clean['province'].groupby(df_clean['province']).size().sort_values(ascending=False))

province
Hà Nội         627
Hồ Chí Minh    178
Đà Nẵng         20
Bắc Ninh         7
Đồng Nai         6
Khánh Hòa        4
Thái Nguyên      4
Tây Ninh         3
Hải Phòng        3
Nhật Bản         3
Quảng Trị        2
Gia Lai          1
Hưng Yên         1
Ninh Bình        1
Nghệ An          1
Lâm Đồng         1
Phú Thọ          1
Quảng Ninh       1
Vĩnh Long        1
Đắk Lắk          1
Name: province, dtype: int64


In [166]:
# Xử lý các dòng có giá trị province không thuộc danh sách các tỉnh thành Việt Nam
vietnam_provinces = [
    "Hà Nội", "Hải Phòng", "Huế", "Đà Nẵng", "Hồ Chí Minh", "Cần Thơ",
    "Cao Bằng", "Điện Biên", "Lai Châu", "Sơn La", "Lạng Sơn",
    "Quảng Ninh", "Thanh Hóa", "Nghệ An", "Hà Tĩnh",
    "Tuyên Quang", "Lào Cai", "Thái Nguyên", "Phú Thọ", "Bắc Ninh",
    "Hưng Yên", "Ninh Bình", "Quảng Trị", "Quảng Ngãi", "Gia Lai",
    "Khánh Hòa", "Lâm Đồng", "Đắk Lắk", "Đồng Nai", "Tây Ninh",
    "Vĩnh Long", "Đồng Tháp", "Cà Mau", "An Giang",
]
print(f"Số lượng dòng trước khi lọc: {len(df_clean)}")
is_vietnam =  df_clean["province"].isin(vietnam_provinces)

# In các dòng không khớp để kiểm tra
display(df_clean.loc[
    ~is_vietnam, ["job_title", "province", "source_url"]
])

# Chỉ giữ các dòng khớp
df_clean = df_clean.loc[is_vietnam].copy()
print(f"Số lượng dòng sau khi lọc: {len(df_clean)}")

Số lượng dòng trước khi lọc: 866


,job_title,province,source_url
174,Kỹ Sư Cầu Nối ( BrSE ),Nhật Bản,https://www.topcv.vn/viec-lam/ky-su-cau-noi-br...
279,C# Developer,Nhật Bản,https://www.topcv.vn/viec-lam/c-developer/1805...
314,Kỹ Sư Cầu Nối,Nhật Bản,https://www.topcv.vn/viec-lam/ky-su-cau-noi/23...


Số lượng dòng sau khi lọc: 863


In [167]:
# Đếm tần suất các giá trị trong cột 'experience_raw', bao gồm cả giá trị NaN
print(df_clean['experience_raw'].value_counts(dropna=False))

experience_raw
3 năm            215
2 năm            213
1 năm            144
5 năm             77
Không yêu cầu     62
4 năm             57
Trên 5 năm        55
Dưới 1 năm        40
Name: count, dtype: int64


    Khảo sát cột `experience_raw` ta thấy có 8 loại kinh nghiệm và có sự phân chia hợp lý gồm:
1. `Dưới 1 năm`
2. `1 năm`
3. `2 năm`
4. `3 năm`
5. `4 năm`
6. `5 năm`
7. `Trên 5 năm`
8. `Không yêu cầu`

    ===> Không cần chuẩn hóa lại cột `experience_raw`.

In [168]:
df_clean = df_clean.rename(columns={'experience_raw': 'experience'}) 

#### Phần khảo sát dữ liệu có trong các văn bản mô tả, yêu cầu và quyền lợi

In [169]:
i=2
description = df_clean.iloc[i]['description_raw']
text = BeautifulSoup(description, 'html.parser').get_text('\n', strip=True)
print(text)

Chịu trách nhiệm phát triển các tính năng mới và bảo trì các tính năng đã phát triển
Thực hiện tất cả các hoạt động kiểm thử (thủ công và tự động) nhằm nâng cao chất lượng sản phẩm. Phối hợp chặt chẽ với các bộ phận liên quan (Lập trình viên, Chuyên viên phân tích nghiệp vụ, Chăm sóc khách hàng, Vận hành, v.v.) để đảm bảo triển khai sản phẩm thành công.
Phân tích và giải quyết các yêu cầu về kiểm thử tự động một cách có phương pháp, tỉ mỉ và có hệ thống, vận dụng kỹ năng phân tích và khả năng giải quyết vấn đề hiệu quả.
Tự động hóa các bài kiểm thử chức năng (Functional Testing), kiểm thử hồi quy (Regression Testing) và/hoặc kiểm thử chấp nhận về hiệu năng (Performance Acceptance Testing).
Chịu trách nhiệm toàn diện trong việc phát triển và mở rộng phạm vi kiểm thử tự động xuyên suốt hệ thống (End-to-End Automated Testing).
Tham gia với nhóm BA để lấy và phân tích yêu cầu từ khách hàng, bàn giao nghiệp vụ lại cho đội phát triển và đồng thời dựa vào đó đưa ra các đề xuất kiểm thử phù hợ

In [170]:
requirement = df_clean.iloc[i]['requirements_raw']
text = BeautifulSoup(requirement, 'html.parser').get_text('\n', strip=True)
print(text)

Yêu cầu ứng viên
●
Kiến thức & Kinh nghiệm:
Có bằng kỹ sư hoặc cử nhân thuộc các chuyên ngành liên quan như Khoa Học Máy Tính, Công Nghệ Thông Tin.
Có ít nhất 2 năm kinh nghiệm thực tế trong lĩnh vực kiểm thử phần mềm.
Có khả năng sử dụng AI Agents để thực hiện các công việc kiểm thử. Cụ thể như sau:
Chuẩn bị ngữ cảnh tốt, rõ ràng, đầy đủ cho AI.
Định hướng, giao nhiệm vụ và điều phối AI Agent thực hiện công việc.
Kiểm tra, đánh giá kết quả kiểm thử do AI tạo ra.
Đảm bảo phần mềm đạt chất lượng sau quá trình kiểm thử.
Có kiến thức vững chắc về quy trình phát triển phần mềm, đặc biệt là quy trình kiểm thử phần mềm.
Có kinh nghiệm chuyên sâu về kiểm thử thủ công (Manual Testing), bao gồm:
Phân tích yêu cầu (Requirement Analysis).
Xây dựng chiến lược kiểm thử (Test Strategy).
Xác định phương pháp tiếp cận kiểm thử (Test Approach).
Lập kế hoạch kiểm thử (Test Plan).
Áp dụng các kỹ thuật kiểm thử như kiểm thử hộp đen (Black-box Testing), kiểm thử dựa trên rủi ro (Risk-based Testing), kiểm t

In [171]:
benefit = df_clean.iloc[i]['benefits_raw']
text = BeautifulSoup(benefit, 'html.parser').get_text('\n', strip=True)
print(text)

Quyền lợi ứng viên
Mức lương cứng: 10 - 15 triệu / tháng (thoả thuận theo năng lực)
Tổng thu nhập ổn định từ: 15-20 triệu / tháng, bao gồm: Lương cứng + Thưởng KPI theo dự án, chiến dịch.
Được hưởng đầy đủ chế độ BHYT, BHXH theo quy định sau khi ký HĐLĐ chính thức.
Chế độ phúc lợi đầy đủ: Quà tặng sinh nhật, thưởng lễ, Tết, hiếu, hỉ, Trung thu, 8/3, 20/10 theo quy chế của Công ty và Công đoàn, khám sức khỏe định kỳ
Thưởng hàng năm từ 2-3 tháng lương tuỳ tình hình lợi nhuân công ty.
Đánh giá xem xét tăng lương định kỳ 2 lần 1 năm.
Làm việc trực tiếp với CTO có hơn 15 năm kinh nghiệm làm việc tại Hàn Quốc thuộc dự án AI của chính phủ. Được tham gia các khóa đào tạo nâng cao nghiệp vụ phục vụ cho công việc.
Môi trường làm việc năng động, trẻ trung, có nhiều cơ hội thăng tiến.
Phụ cấp: đồng phục, điện thoại.


In [172]:
# Chuẩn bị stopwords cho cả tiếng Việt và tiếng Anh
stop_words = set(stopwords("vi")) | set(stopwords("en"))

stop_words -= {
    "không", "chưa", "chẳng",
    "not", "no", "never",
    "c", "r", "go"
}

# Làm sạch HTML và loại stopwords
def clean_text(html):
    if pd.isna(html):
        return ""

    text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
    return " ".join(
        word for word in text.lower().split()
        if word.strip(".,;:!?()[]") not in stop_words
    )

# Làm sạch các cột văn bản
for field in ["description", "requirements", "benefits"]:
    df_clean[f"{field}_text"] = df_clean[f"{field}_raw"].apply(clean_text)

In [173]:
# Trích 2 mảng kỹ năng cần có và kỹ năng nên có từ tag 
def extract_skills(html):
    result = {
        "required_skills": [],
        "preferred_skills": [],
    }

    labels = {
        "kỹ năng cần có": "required_skills",
        "kỹ năng nên có": "preferred_skills",
        "required skills": "required_skills",
        "preferred skills": "preferred_skills",
    }

    if pd.notna(html):
        soup = BeautifulSoup(html, "html.parser")

        for tag in soup.select(".required-tag"):
            title = tag.select_one(".required-tag__content--title")
            content = tag.select_one(".required-tag__content--desc")

            if title and content:
                label = title.get_text(" ", strip=True).lower().strip(": ")
                key = labels.get(label)

                if key:
                    result[key].extend(
                        skill.strip().lower()
                        for skill in re.split(
                            r",\s*(?![^()]*\))",
                            content.get_text(" ", strip=True)
                        )
                        if skill.strip()
                    )

    # Bỏ trùng, giữ nguyên tên và thứ tự kỹ năng
    return pd.Series({
        key: list(dict.fromkeys(skills))
        for key, skills in result.items()
    })


skill_columns = ["required_skills", "preferred_skills"]

df_clean[skill_columns] = (
    df_clean["requirements_raw"].apply(extract_skills)
)

display(df_clean[skill_columns].head(10))

,required_skills,preferred_skills
0,"[kiến thức về kiểm thử phần mềm, kỹ năng viết ...","[bug tracking tools (jira, bugzilla), test aut..."
1,[],[]
2,"[git, java, javascript, tester, phân tích, non...",[]
3,[],[]
4,"[javascript, python, ai, typescript, ci cd pip...",[]
5,[],[]
6,"[business analyst, công nghệ thông tin]",[]
7,"[java, javascript framework]",[]
8,"[logic, bug reporting, software testing, respo...","[regression test, boundary value, abnormal tes..."
9,[],[]


===> Sau khi tách tag, một số tin có mảng kỹ năng rỗng. Điều này không có nghĩa tin không yêu cầu kỹ năng; nhà tuyển dụng có thể chỉ viết chúng trong mô tả hoặc nội dung yêu cầu.

 - Nhóm gộp `required_skills` và `preferred_skills`, loại bỏ trùng lặp để tạo cột `skills`, biểu diễn các kỹ năng được đề cập trong tin tuyển dụng, không phân biệt bắt buộc hay ưu tiên.

 - Với những dòng `skills` còn rỗng, nhóm tìm bổ sung trong `description_raw` và `requirements_raw`. Danh sách dùng để tìm được tổng hợp từ các tag kỹ năng đã thu thập. Nếu không tìm thấy kỹ năng phù hợp, giữ nguyên mảng rỗng, không tự suy đoán.

In [174]:
# Tạo danh sách kỹ năng từ 2 cột required_skills và preferred_skills
skill_dictionary = sorted({
    skill
    for column in ["required_skills", "preferred_skills"]
    for skills in df_clean[column]
    for skill in skills
})

# Tạo cột skills từ 2 cột required_skills và preferred_skills, nếu cả 2 cột đều rỗng thì trích từ description_raw và requirements_raw
def collect_skills(row):
    skills = row["required_skills"] + row["preferred_skills"]

    if skills:
        return list(dict.fromkeys(skills))

    if pd.isna(row["requirements_raw"]):
        return []

    text = BeautifulSoup(row["requirements_raw"], "html.parser").get_text(" ", strip=True) + " " + BeautifulSoup(row["description_raw"], "html.parser").get_text(" ", strip=True)

    return [
        skill for skill in skill_dictionary
        if re.search(
            rf"(?<!\w){re.escape(skill)}(?!\w)",
            text,
            flags=re.IGNORECASE
        )
    ]

df_clean["skills"] = df_clean.apply(collect_skills, axis=1)

display(df_clean[
    ["required_skills", "preferred_skills", "skills"]
].head(10))

,required_skills,preferred_skills,skills
0,"[kiến thức về kiểm thử phần mềm, kỹ năng viết ...","[bug tracking tools (jira, bugzilla), test aut...","[kiến thức về kiểm thử phần mềm, kỹ năng viết ..."
1,[],[],"[an ninh mạng, analysis, communication, data, ..."
2,"[git, java, javascript, tester, phân tích, non...",[],"[git, java, javascript, tester, phân tích, non..."
3,[],[],"[.net, ai, ai agent, angular, api, aws, azure,..."
4,"[javascript, python, ai, typescript, ci cd pip...",[],"[javascript, python, ai, typescript, ci cd pip..."
5,[],[],"[a/b testing, ai, ai tools, api, apis, backend..."
6,"[business analyst, công nghệ thông tin]",[],"[business analyst, công nghệ thông tin]"
7,"[java, javascript framework]",[],"[java, javascript framework]"
8,"[logic, bug reporting, software testing, respo...","[regression test, boundary value, abnormal tes...","[logic, bug reporting, software testing, respo..."
9,[],[],"[it, marketing, phần mềm, đào tạo]"


In [175]:
empty_skills = df_clean["skills"].map(len).eq(0)

print("Số dòng chưa tìm được kỹ năng:", empty_skills.sum())
display(df_clean.loc[empty_skills, ["job_title", "source_url"]])

Số dòng chưa tìm được kỹ năng: 0


,job_title,source_url


===> Đã điền đủ các skills vào ô dữ liệu trống

In [176]:
# Xem xét các giá trị trong cột level_raw
print(df_clean["level_raw"].value_counts(dropna=False))
print(f"Tổng số tin tuyển dụng: {len(df_clean)}")

level_raw
Nhân viên             547
Trưởng nhóm           202
Trưởng/Phó phòng       40
Thực tập sinh          37
Quản lý / Giám sát     34
Giám đốc                3
Name: count, dtype: int64
Tổng số tin tuyển dụng: 863


===> Ta thấy tổng có 863 tin tuyển dụng, có 6 cấp bậc và không có giá trị thiếu. Nhóm quyết định giữ các phân loại này để thực hiện các bước phân tích tiếp theo.

In [180]:
# Xuất file dữ liệu sạch
print(f"Kích thước dữ liệu sạch: {df_clean.shape[0]}")
columns_to_export = [
    "job_title",
    "company_name",
    "province",
    "description_text",
    "requirements_text",
    "benefits_text",
    "level_raw",
    "salary_min",
    "salary_max",
    "salary_type",
    "experience",
    "skills",
    "source_url",
    "scraped_at",
]
out_file = PROJECT_ROOT / "data" / "processed" / "clean_data.csv"
df_clean[columns_to_export].to_csv(out_file, index=False, encoding="utf-8-sig")
print(f"Đã xuất: 'clean_data.csv' ==> {out_file.parent.resolve()}")

Kích thước dữ liệu sạch: 863
Đã xuất: 'clean_data.csv' ==> C:\Secret\Khai phá dữ liệu\data-mining\data\processed
